# Pushup Rep Counter - Testing & Evaluation Notebook

This notebook is for testing the pushup counter on video files and evaluating accuracy.

## Setup

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from utils.rep_counter import PushupCounter
import os

## 1. Initialize Counter

In [ ]:
# Create pushup counter with configurable thresholds
counter = PushupCounter(
    angle_threshold_down=90,   # Angle when arms are bent
    angle_threshold_up=150      # Angle when arms are extended
)
print("Pushup counter initialized!")

## 2. Process Video File

In [ ]:
# Path to video file
video_path = "data/test_pushups.mp4"

if not os.path.exists(video_path):
    print(f"Video file not found: {video_path}")
    print("Please place a test video in the data/ folder")
else:
    # Open video
    cap = cv2.VideoCapture(video_path)
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frame_count / fps if fps > 0 else 0
    
    print(f"Video Properties:")
    print(f"  FPS: {fps}")
    print(f"  Total Frames: {frame_count}")
    print(f"  Duration: {duration:.2f} seconds")
    
    # Process video
    angles_log = []
    status_log = []
    
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Resize for faster processing
        frame = cv2.resize(frame, (640, 480))
        
        # Process frame
        result_frame, rep_count, angle, left_angle, right_angle, status = \
            counter.process_frame(frame)
        
        angles_log.append(angle)
        status_log.append(status)
        
        if (frame_idx + 1) % 30 == 0:
            print(f"Frame {frame_idx + 1}/{frame_count} - Reps: {rep_count}, Angle: {angle:.1f}°")
        
        frame_idx += 1
    
    cap.release()
    print(f"\n✅ Video Processing Complete!")
    print(f"Total Reps Detected: {rep_count}")

## 3. Visualize Results

In [ ]:
# Plot angle variations over time
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(angles_log, linewidth=2)
plt.axhline(y=90, color='r', linestyle='--', label='Down Threshold (90°)')
plt.axhline(y=150, color='g', linestyle='--', label='Up Threshold (150°)')
plt.xlabel('Frame Number')
plt.ylabel('Elbow Angle (degrees)')
plt.title('Elbow Angle Over Time')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot status changes
plt.subplot(1, 2, 2)
status_numeric = [0 if s == 'Up' else 1 for s in status_log]
plt.fill_between(range(len(status_log)), status_numeric, alpha=0.5)
plt.xlabel('Frame Number')
plt.ylabel('Position')
plt.title('Pushup Position Over Time')
plt.yticks([0, 1], ['Up', 'Down'])
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAngle Statistics:")
print(f"  Min: {min(angles_log):.1f}°")
print(f"  Max: {max(angles_log):.1f}°")
print(f"  Mean: {np.mean(angles_log):.1f}°")
print(f"  Std Dev: {np.std(angles_log):.1f}°")

## 4. Manual Verification

Enter the actual number of pushups in the video and compare with detected count.

In [ ]:
# Manual count from video
actual_reps = 10  # Change this to actual count
detected_reps = rep_count

accuracy = abs(actual_reps - detected_reps) / actual_reps * 100

print(f"Actual Reps: {actual_reps}")
print(f"Detected Reps: {detected_reps}")
print(f"Difference: {abs(actual_reps - detected_reps)}")
print(f"Accuracy: {100 - accuracy:.1f}% (if perfect would be 100%)")

## 5. Threshold Tuning

If accuracy is not satisfactory, adjust thresholds and reprocess.

In [ ]:
# Try different thresholds
for down_thresh in [80, 85, 90, 95, 100]:
    for up_thresh in [145, 150, 155, 160]:
        counter_test = PushupCounter(
            angle_threshold_down=down_thresh,
            angle_threshold_up=up_thresh
        )
        
        # Quick test - process first 5 frames
        # (For full test, load video again)
        
        print(f"Thresholds: Down={down_thresh}°, Up={up_thresh}° (Placeholder)")